In [1]:
!pip install meteostat

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
from meteostat import Point, Daily
from datetime import datetime
import os
import time
import random
from urllib.error import URLError


MATCH_CSV = "data_processed/venue_match_level.csv"
CACHE_CSV = "data_processed/climate_match_cache.csv"
OUT_CSV   = "data_processed/match_with_climate.csv"


MAX_RETRIES = 3
BASE_SLEEP = 0.6
MAX_TIME_PER_MATCH = 15    
COORD_ROUND = 4
BATCH_SAVE_SIZE = 50        


df = pd.read_csv(MATCH_CSV)

df["match_date"] = pd.to_datetime(df["match_date"], errors="coerce").dt.date
df = df.dropna(subset=["match_id", "latitude", "longitude", "match_date"])

df["latitude"]  = df["latitude"].round(COORD_ROUND)
df["longitude"] = df["longitude"].round(COORD_ROUND)
df["match_id"]  = df["match_id"].astype(str)


if os.path.exists(CACHE_CSV):
    cache = pd.read_csv(CACHE_CSV)
    cache["match_id"] = cache["match_id"].astype(str)
else:
    cache = pd.DataFrame(columns=[
        "match_id",
        "latitude", "longitude", "match_date",
        "tavg", "tmax", "tmin",
        "rhum", "prcp", "wspd", "pres", "tsun"
    ])


pending = df.loc[~df["match_id"].isin(cache["match_id"])].copy()

print(f" Climate fetch required for {len(pending)} matches")


new_rows = []

for i, r in enumerate(pending.itertuples(index=False), 1):
    point = Point(r.latitude, r.longitude)
    start = datetime.combine(r.match_date, datetime.min.time())
    end = start

    data = None
    t0 = time.time()

    for attempt in range(MAX_RETRIES):
        if time.time() - t0 > MAX_TIME_PER_MATCH:
            break

        try:
            data = Daily(point, start, end).fetch()
            break
        except (URLError, Exception):
            sleep = BASE_SLEEP * (2 ** attempt) + random.uniform(0, 0.5)
            time.sleep(sleep)

    row = {
        "match_id": r.match_id,
        "latitude": r.latitude,
        "longitude": r.longitude,
        "match_date": r.match_date
    }

    if data is not None and not data.empty:
        w = data.iloc[0]
        row.update({
            "tavg": w.get("tavg"),
            "tmax": w.get("tmax"),
            "tmin": w.get("tmin"),
            "rhum": w.get("rhum"),
            "prcp": w.get("prcp"),
            "wspd": w.get("wspd"),
            "pres": w.get("pres"),
            "tsun": w.get("tsun")
        })

    new_rows.append(row)
    time.sleep(BASE_SLEEP)


    if i % BATCH_SAVE_SIZE == 0 or i == len(pending):
        pd.DataFrame(new_rows).to_csv(
            CACHE_CSV,
            mode="a",
            header=not os.path.exists(CACHE_CSV),
            index=False
        )
        new_rows.clear()
        print(f"Saved {i}/{len(pending)} climate rows")

cache = pd.read_csv(CACHE_CSV)
cache["match_id"] = cache["match_id"].astype(str)

df = df.merge(
    cache,
    on=["match_id", "latitude", "longitude", "match_date"],
    how="left"
)


df["dew_risk"] = (df["rhum"] >= 70).astype(int)

df["swing_friendly"] = (
    (df["rhum"] >= 60) &
    (df["wspd"] >= 8) &
    (df["pres"] <= 1010)
).astype(int)

df["heat_stress"] = (df["tmax"] - 30).clip(lower=0)
df["hot_day"] = (df["tmax"] >= 32).astype(int)

df["rain_risk"] = (df["prcp"] > 0).astype(int)
df["rain_heavy"] = (df["prcp"] >= 5).astype(int)

df["windy"] = (df["wspd"] >= 15).astype(int)
df["low_pressure"] = (df["pres"] <= 1005).astype(int)

df["batting_friendly_score"] = (
    (df["tavg"] / 40).fillna(0) +
    ((100 - df["rhum"]) / 100).fillna(0) +
    (1 - df["windy"])
) / 3

df["bowling_friendly_score"] = (
    0.3 * df["dew_risk"] +
    0.4 * df["swing_friendly"] +
    0.3 * df["low_pressure"]
)

df["batting_friendly_score"] = df["batting_friendly_score"].clip(0, 1)
df["bowling_friendly_score"] = df["bowling_friendly_score"].clip(0, 1)


df.to_csv(OUT_CSV, index=False)

print(" Climate extraction (match-level cache) complete")
print(f"Total matches: {len(df)}")
print(f"Cached matches: {len(cache)}")


🌦️ Climate fetch required for 7967 matches
Saved 50/7967 climate rows
Saved 100/7967 climate rows
Saved 150/7967 climate rows
Saved 200/7967 climate rows
Saved 250/7967 climate rows
Saved 300/7967 climate rows
Saved 350/7967 climate rows
Saved 400/7967 climate rows
Saved 450/7967 climate rows
Saved 500/7967 climate rows
Saved 550/7967 climate rows
Saved 600/7967 climate rows
Saved 650/7967 climate rows
Saved 700/7967 climate rows
Saved 750/7967 climate rows
Saved 800/7967 climate rows
Saved 850/7967 climate rows
Saved 900/7967 climate rows
Saved 950/7967 climate rows
Saved 1000/7967 climate rows
Saved 1050/7967 climate rows
Saved 1100/7967 climate rows
Saved 1150/7967 climate rows
Saved 1200/7967 climate rows
Saved 1250/7967 climate rows
Saved 1300/7967 climate rows
Saved 1350/7967 climate rows
Saved 1400/7967 climate rows
Saved 1450/7967 climate rows
Saved 1500/7967 climate rows


Saved 1550/7967 climate rows
Saved 1600/7967 climate rows
Saved 1650/7967 climate rows
Saved 1700/7967 climate rows
Saved 1750/7967 climate rows
Saved 1800/7967 climate rows
Saved 1850/7967 climate rows
Saved 1900/7967 climate rows
Saved 1950/7967 climate rows
Saved 2000/7967 climate rows
Saved 2050/7967 climate rows
Saved 2100/7967 climate rows
Saved 2150/7967 climate rows
Saved 2200/7967 climate rows
Saved 2250/7967 climate rows
Saved 2300/7967 climate rows
Saved 2350/7967 climate rows
Saved 2400/7967 climate rows
Saved 2450/7967 climate rows
Saved 2500/7967 climate rows
Saved 2550/7967 climate rows
Saved 2600/7967 climate rows
Saved 2650/7967 climate rows
Saved 2700/7967 climate rows
Saved 2750/7967 climate rows
Saved 2800/7967 climate rows
Saved 2850/7967 climate rows
Saved 2900/7967 climate rows
Saved 2950/7967 climate rows
Saved 3000/7967 climate rows
Saved 3050/7967 climate rows
Saved 3100/7967 climate rows
Saved 3150/7967 climate rows
Saved 3200/7967 climate rows
Saved 3250/796

Saved 5850/7967 climate rows
Saved 5900/7967 climate rows
Saved 5950/7967 climate rows
Saved 6000/7967 climate rows
Saved 6050/7967 climate rows
Saved 6100/7967 climate rows
Saved 6150/7967 climate rows
Saved 6200/7967 climate rows
Saved 6250/7967 climate rows
Saved 6300/7967 climate rows
Saved 6350/7967 climate rows
Saved 6400/7967 climate rows
Saved 6450/7967 climate rows
Saved 6500/7967 climate rows
Saved 6550/7967 climate rows
Saved 6600/7967 climate rows
Saved 6650/7967 climate rows
Saved 6700/7967 climate rows
Saved 6750/7967 climate rows
Saved 6800/7967 climate rows
Saved 6850/7967 climate rows
Saved 6900/7967 climate rows
Saved 6950/7967 climate rows
Saved 7000/7967 climate rows
Saved 7050/7967 climate rows
Saved 7100/7967 climate rows
Saved 7150/7967 climate rows
Saved 7200/7967 climate rows
Saved 7250/7967 climate rows
Saved 7300/7967 climate rows
Saved 7350/7967 climate rows
Saved 7400/7967 climate rows
Saved 7450/7967 climate rows
Saved 7500/7967 climate rows
Saved 7550/796

In [ ]:
import pandas as pd
import numpy as np


CACHE_CSV = "data_processed/climate_match_cache.csv"
OUT_CSV   = "data_processed/match_with_climate_final.csv"


df = pd.read_csv(CACHE_CSV)


df["match_date"] = pd.to_datetime(df["match_date"], errors="coerce")

df["tavg"] = df["tavg"].fillna(
    (df["tmax"] + df["tmin"]) / 2
)


GROUP_COLS = ["latitude", "longitude"]

def venue_fill(col, default=None):
    df[col] = (
        df.groupby(GROUP_COLS)[col]
          .transform(lambda x: x.fillna(x.median()))
    )
    if default is not None:
        df[col] = df[col].fillna(default)

venue_fill("rhum", default=65)    # typical cricket humidity
venue_fill("wspd", default=10)    # light breeze
venue_fill("pres", default=1013)  # standard pressure


df["prcp"] = df["prcp"].fillna(0)


raw_cols = ["tavg", "tmax", "tmin", "rhum", "prcp", "wspd", "pres"]

df["climate_fields_present"] = df[raw_cols].notna().sum(axis=1)
df["climate_reliability"] = df["climate_fields_present"] / len(raw_cols)



# Heat
df["hot_day"] = (df["tavg"] >= 30).astype(int)
df["very_hot_day"] = (df["tavg"] >= 35).astype(int)

# Rain / interruption risk
df["rain_risk"] = (df["prcp"] > 0).astype(int)
df["heavy_rain_risk"] = (df["prcp"] >= 5).astype(int)

# Dew & swing
df["humid_day"] = (df["rhum"] >= 70).astype(int)

df["swing_friendly"] = (
    (df["rhum"] >= 60) &
    (df["wspd"] >= 8) &
    (df["pres"] <= 1010)
).astype(int)

# Wind
df["windy_day"] = (df["wspd"] >= 15).astype(int)



# Batting-friendly conditions
df["batting_friendly_score"] = (
    0.4 * (df["tavg"] / 40).clip(0, 1) +
    0.3 * (1 - (df["rhum"] / 100)) +
    0.3 * (1 - df["windy_day"])
)

# Bowling-friendly conditions
df["bowling_friendly_score"] = (
    0.4 * df["swing_friendly"] +
    0.3 * df["humid_day"] +
    0.3 * (1 - df["hot_day"])
)

df["batting_friendly_score"] = df["batting_friendly_score"].clip(0, 1)
df["bowling_friendly_score"] = df["bowling_friendly_score"].clip(0, 1)


df = df.drop(columns=["climate_fields_present"])


df.to_csv(OUT_CSV, index=False)

print(" Climate post-processing complete")
print(f"Rows processed: {len(df)}")
print("Output saved to:", OUT_CSV)


✅ Climate post-processing complete
Rows processed: 7967
Output saved to: data_processed/match_with_climate_final.csv
